## Secure Credential Management

# Introduction: The Importance of Secure Credentials

Welcome to the first lesson of the **Developer Security & Observability** course. In this lesson, we will focus on secure credentials management using **Google Cloud Secret Manager** and the Google Cloud client library for Python.

Credentials—such as usernames, passwords, or API keys—allow applications to access critical resources like databases or external services. If these credentials are not protected, they can be stolen or misused, leading to catastrophic security breaches.

Google Cloud provides two primary tools for this:
* **Google Cloud Secret Manager:** Securely stores, manages, and retrieves secrets.
* **Google Cloud Client Library:** Allows your applications to interact with GCP services using authenticated requests.

By the end of this lesson, you will know how to store a secret in Secret Manager and retrieve it securely using the Python client library.

---

## Understanding GCP Credential Management

When you use a Google Cloud client library, it needs credentials to authenticate your requests. The library automatically looks for credentials in the following order (known as the **Credential Chain**):

1.  **Environment variables** (`GOOGLE_APPLICATION_CREDENTIALS`)
2.  **Application Default Credentials (ADC)** configured on your local machine.
3.  **Service accounts** (when running on GCP services like Compute Engine, Cloud Run, or Cloud Functions).

> ### 🚀 Production Best Practice
> In production environments running on Google Cloud, you should use **Service Accounts** instead of static credentials. Service accounts provide automatic, temporary credentials, eliminating the need to store long-term access keys in your code or config files.

---

## Storing Secrets with Secret Manager

### Step 1: Import Required Libraries
We use `google.cloud.secretmanager` to interact with the service and `json` to format our sensitive data.

```python
import json
from google.cloud import secretmanager

# Initialize the client
client = secretmanager.SecretManagerServiceClient()
```

### Step 2: Define the Secret
Define your target Project ID and the payload you wish to hide.

```python
PROJECT_ID = "your-project-id"
SECRET_ID = "app-db-credentials"

payload = {
    "username": "appuser",
    "password": "S3cureP@ss!",
    "host": "db.example.com"
}
```

> [!WARNING]
> **Security Warning:** The credentials shown above are for illustration only. Never commit real credentials to version control (Git), configuration files, or application logs.

### Step 3: Store the Secret
Storing a secret involves creating the **Secret container** and then adding a **Secret Version** containing the actual data.

```python
# 1. Create the secret container with an Automatic replication policy
parent = f"projects/{PROJECT_ID}"
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": SECRET_ID,
        "secret": {"replication": {"automatic": {}}},
    }
)

# 2. Add the secret data as a new version
secret_data = json.dumps(payload).encode("UTF-8")
version = client.add_secret_version(
    request={"parent": secret.name, "payload": {"data": secret_data}}
)

print(f"Created: {version.name}")
```
**Example Output:**
`Created: projects/123456789012/secrets/app-db-credentials/versions/1`

---

## Retrieving Secrets in Your Application

The retrieval flow ensures that no secrets are hardcoded in your application code.



### Step 1: Define a Function to Retrieve the Secret
We point the client to the `latest` version to ensure we always get the most current credentials.

```python
def get_secret(project_id, secret_id):
    # Point to the 'latest' version
    name = f"projects/{project_id}/secrets/{secret_id}/versions/latest"
    
    # Access the version
    response = client.access_secret_version(request={"name": name})
    
    # Decode and parse
    secret_data = response.payload.data.decode("UTF-8")
    return json.loads(secret_data)
```

### Step 2: Access the Secret Data
Now, let's use the function and access a non-sensitive field like the `host`.

```python
if __name__ == "__main__":
    secret = get_secret(PROJECT_ID, "app-db-credentials")
    print("Secret host:", secret["host"])
    
    # Use secret["password"] for your DB connection logic
    # BUT NEVER PRINT IT!
```

---

## Production Deployment: Service Accounts

In production, your application (running on Compute Engine or Cloud Functions) uses an attached Service Account. This provides temporary credentials via the **Instance Metadata Service**.



### Key Advantages:
* **No static credentials:** No keys to leak from config files.
* **Automatic rotation:** Google handles the lifecycle of the temporary credentials.
* **Identical code:** The client library code you wrote above remains exactly the same; it discovers the Service Account automatically!

---

## Summary and What's Next

In this lesson, you learned:
* **Why** secure credential management is vital for cloud security.
* **How** to use Secret Manager to create and version sensitive data.
* **The Credential Chain** and how client libraries authenticate.
* **The Best Practice** of using Service Accounts in production.

In the next practice exercises, you will get hands-on experience writing Python code to manage secrets yourself. Build securely from the start!

## Update the Secret Payload with Port and New Password

Now that you understand how to store and retrieve secrets with Google Cloud Secret Manager, let's practice working with the payload data structure itself. You'll modify an existing payload dictionary to include additional database connection information and update the password for better security.

Your task is to update the payload dictionary in main.py with two changes:

    Add a new port field with the value "5432" (the default PostgreSQL port).
    Change the password from its current value to NewS3cure123!.

Keep all other code unchanged — you only need to update the payload dictionary. The other files will work automatically with your updated credentials.

Note: The code is designed to be idempotent, meaning you can run it multiple times safely. If the secret already exists, it will be updated instead of causing an error.

This hands-on practice will help you understand how different types of credential information can be structured and stored securely in Google Cloud Secret Manager.

```python
import json
from google.auth.credentials import AnonymousCredentials
from google.api_core.client_options import ClientOptions
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient(
    credentials=AnonymousCredentials(),
    client_options=ClientOptions(api_endpoint="http://localhost:8080"),
    transport="rest",
)

SECRET_NAME = "projects/demo-project/secrets/app-db-credentials"
# TODO: Add a "port" field with value "5432" and change the password to "NewS3cure123!"
payload = {"username": "appuser", "password": "S3cureP@ss!", "host": "db.example.com"}

def create_or_update_secret():
    parent = SECRET_NAME
    try:
        client.create_secret(
            parent="/".join(parent.split("/")[:-2]),
            secret_id=parent.split("/")[-1],
            secret={"replication": {"automatic": {}}}
        )
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Created:", parent)
    except Exception as e:
        # Assume secret exists, so add a new version
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Updated:", parent)

if __name__ == "__main__":
    create_or_update_secret()
```

Updating a secret's payload is a frequent task in the development lifecycle, especially when rotating passwords or adding configuration details like ports. In this case, adding the `port` field and hardening the `password` ensures your application has all the necessary (and secure) metadata for a database connection.

---

### Updated `main.py`

Update the `payload` dictionary as follows:

```python
import json
from google.auth.credentials import AnonymousCredentials
from google.api_core.client_options import ClientOptions
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient(
    credentials=AnonymousCredentials(),
    client_options=ClientOptions(api_endpoint="http://localhost:8080"),
    transport="rest",
)

SECRET_NAME = "projects/demo-project/secrets/app-db-credentials"

# FIX: Added "port" and updated "password"
payload = {
    "username": "appuser", 
    "password": "NewS3cure123!", 
    "host": "db.example.com",
    "port": "5432"
}

def create_or_update_secret():
    parent = SECRET_NAME
    try:
        # Attempt to create the secret container first
        client.create_secret(
            parent="/".join(parent.split("/")[:-2]),
            secret_id=parent.split("/")[-1],
            secret={"replication": {"automatic": {}}}
        )
        # Create the first version
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Created:", parent)
    except Exception as e:
        # If the secret already exists, this block runs to add a new version
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Updated:", parent)

if __name__ == "__main__":
    create_or_update_secret()
```

---

### Why this approach works:

*   **JSON Serialization**: By storing the data as a dictionary and using `json.dumps()`, you make it easy for your application to retrieve the secret and convert it back into a Python object with a single line of code.
*   **Immutability of Versions**: Secret Manager doesn't "edit" an existing version. Instead, the `add_secret_version` method creates a new, immutable version of the secret. Your application can then point to the `latest` version to get these updated credentials automatically.
*   **Idempotency**: The `try...except` block ensures that your automation script won't fail if it's run twice. It handles the "Secret Already Exists" error by simply appending the new version to the existing secret container.

## Parsing Secret Payloads from Google Cloud Secret Manager


When Google Cloud Secret Manager returns a secret, the secret payload is delivered as a JSON string, but your application code needs to access individual fields like host or password. This means you need to convert that JSON string into a Python dictionary for key-based access.

Your task is to fix the get_secret function in main.py. Currently, the function returns the raw JSON string, but the code that retrieves and uses the secret expects a dictionary so it can access secret["host"]. Look for the TODO comment and add the missing line that parses the JSON string into a dictionary.

This exercise will help you understand that data retrieved from external services often needs format conversion before your application can use it effectively.

```python
import json
from google.auth.credentials import AnonymousCredentials
from google.api_core.client_options import ClientOptions
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient(
    credentials=AnonymousCredentials(),
    client_options=ClientOptions(api_endpoint="http://localhost:8080"),
    transport="rest",
)

PROJECT_ID = "demo-project"
SECRET_ID = "app-db-credentials"
SECRET_NAME = f"projects/{PROJECT_ID}/secrets/{SECRET_ID}"
payload = {"username": "appuser", "password": "S3cureP@ss!", "host": "db.example.com"}

def get_secret(secret_name):
    response = client.access_secret_version(name=secret_name + "/versions/latest")
    data = response.payload.data.decode("UTF-8")
    # TODO: Add the line that converts the JSON string to a Python dictionary
    return data

def create_secret():
    parent = f"projects/{PROJECT_ID}"
    try:
        client.create_secret(
            parent=parent,
            secret_id=SECRET_ID,
            secret={"replication": {"automatic": {}}}
        )
    except Exception:
        pass  # Secret may already exist

    version = client.add_secret_version(
        parent=SECRET_NAME,
        payload={"data": json.dumps(payload).encode("UTF-8")}
    )
    print("Created:", SECRET_NAME)

if __name__ == "__main__":
    # Create the secret first
    create_secret()
    
    # Retrieve and use the secret
    secret = get_secret(SECRET_NAME)
    print("Secret host:", secret["host"])

```

To make your application functional, you need to bridge the gap between a **JSON string** (text) and a **Python dictionary** (data structure). 

When Secret Manager returns the payload, the `decode("UTF-8")` method turns the raw bytes into a string that *looks* like a dictionary, but Python still treats it as a single block of text. By adding `json.loads()`, you instruct Python to parse that text and transform it into an object where you can access specific keys like `host` or `password`.

### Fixed `main.py`

Here is the corrected `get_secret` function:

```python
def get_secret(secret_name):
    # Retrieve the latest version of the secret
    response = client.access_secret_version(name=secret_name + "/versions/latest")
    
    # Decode the byte-string into a regular UTF-8 string
    data = response.payload.data.decode("UTF-8")
    
    # FIX: Convert the JSON string into a Python dictionary
    return json.loads(data)
```

---

### Understanding the Data Transformation Flow



The transformation happens in three distinct stages:

1.  **Raw Bytes (`bytes`)**: The format Secret Manager uses for storage and transmission. 
    *   *Example:* `b'{"host": "db.example.com"}'`
2.  **UTF-8 String (`str`)**: After calling `.decode("UTF-8")`, it becomes a readable string, but you still cannot access `data["host"]`.
    *   *Example:* `'{"host": "db.example.com"}'`
3.  **Python Dictionary (`dict`)**: After calling `json.loads(data)`, it becomes a key-value object.
    *   *Example:* `{"host": "db.example.com"}`

### Why this is essential
In production environments, your secrets will often contain multiple fields (username, password, port, host). Manually searching through a string to find these values is prone to errors. Using `json.loads()` ensures that your application retrieves the exact value associated with the key, making your code cleaner and more robust.

## Use Constants for Secret Names Instead of Hardcoded Strings

Now let's practice a key coding habit for secure secret management: using constants instead of hardcoded strings.

In the code, a SECRET_NAME constant is defined at the top of main.py, but the code ignores it and uses a different hardcoded string when creating the secret. This mismatch causes problems because other parts of the application expect to find the secret using the constant name.

Your task is to find where the secret is being created and replace the hardcoded string with the SECRET_NAME constant. If you run the code before fixing it, it will fail because the secret is created with one name but retrieved with another.

Note: The code is designed to be idempotent, meaning you can run it multiple times safely. If the secret already exists, it will be updated instead of causing an error.

This exercise will help you understand why consistent naming is crucial for maintainable and reliable applications.

```python
import json
from google.auth.credentials import AnonymousCredentials
from google.api_core.client_options import ClientOptions
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient(
    credentials=AnonymousCredentials(),
    client_options=ClientOptions(api_endpoint="http://localhost:8080"),
    transport="rest",
)

SECRET_NAME = "projects/PROJECT_ID/secrets/app-db-credentials"
payload = {"username": "appuser", "password": "S3cureP@ss!", "host": "db.example.com"}

def create_or_update_secret():
    # TODO: Replace the hardcoded string with the SECRET_NAME constant
    parent = "projects/PROJECT_ID/secrets/hardcoded-secret-name"
    try:
        client.create_secret(
            parent="/".join(parent.split("/")[:-2]),
            secret_id=parent.split("/")[-1],
            secret={"replication": {"automatic": {}}}
        )
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Created:", parent)
    except Exception as e:
        # Assume secret exists, so add a new version
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Updated:", parent)

if __name__ == "__main__":
    create_or_update_secret()

```

Using constants for resource names is a foundational best practice in cloud development. It ensures that your creation, retrieval, and update logic all point to the exact same resource, preventing "Resource Not Found" errors that stem from simple typos or mismatched strings.

In your current code, the `create_or_update_secret` function was targeting `hardcoded-secret-name`, while the rest of your application was likely looking for `app-db-credentials`. By switching to the `SECRET_NAME` constant, you unify these operations.

### Fixed `main.py`

Here is the corrected code with the constant implemented:

```python
import json
from google.auth.credentials import AnonymousCredentials
from google.api_core.client_options import ClientOptions
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient(
    credentials=AnonymousCredentials(),
    client_options=ClientOptions(api_endpoint="http://localhost:8080"),
    transport="rest",
)

SECRET_NAME = "projects/PROJECT_ID/secrets/app-db-credentials"
payload = {"username": "appuser", "password": "S3cureP@ss!", "host": "db.example.com"}

def create_or_update_secret():
    # FIX: Replaced the hardcoded string with the SECRET_NAME constant
    parent = SECRET_NAME
    
    try:
        # Create the secret container
        client.create_secret(
            parent="/".join(parent.split("/")[:-2]),
            secret_id=parent.split("/")[-1],
            secret={"replication": {"automatic": {}}}
        )
        # Add the first version
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Created:", parent)
    except Exception as e:
        # If the secret already exists, we simply add a new version to the existing container
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Updated:", parent)

if __name__ == "__main__":
    create_or_update_secret()
```

---

### Why Consistency Matters in GCP

When you manage secrets (or any cloud resource), you are dealing with **Resource Names**. These act like a unique URL for your data.



1.  **Avoid Discovery Issues**: If your code creates a secret called `app-db-prod` but your deployment script looks for `database-secret`, the deployment will fail.
2.  **Centralized Configuration**: If you ever need to change the name of your secret (e.g., adding a environment suffix like `-dev`), you only have to change it in one place (the constant) rather than hunting through dozens of functions.
3.  **Readability**: Using `parent = SECRET_NAME` makes the intent of the code immediately clear to other developers, whereas a long hardcoded string is "magic text" that is harder to interpret at a glance.

## Handle missing secrets gracefully with Google Cloud Secret Manager

Great job on following naming conventions! Now, let's focus on a critical aspect of working with external services like Google Cloud: error handling.

When your application tries to retrieve a secret that doesn't exist, Google Cloud Secret Manager will raise an exception that can crash your program. This is a real-world scenario — maybe the secret was deleted, or there's a typo in the secret name.

Your task is to add proper exception handling to the get_secret function in main.py:

    Import NotFound from google.api_core.exceptions at the top of the file
    Wrap the client.access_secret_version() call in a try-except block
    Catch NotFound exceptions and print "Secret not found"
    Return an empty dictionary {} when a secret is not found


```python
import json
from google.auth.credentials import AnonymousCredentials
from google.api_core.client_options import ClientOptions
from google.cloud import secretmanager
# TODO: Import NotFound from google.api_core.exceptions

client = secretmanager.SecretManagerServiceClient(
    credentials=AnonymousCredentials(),
    client_options=ClientOptions(api_endpoint="http://localhost:8080"),
    transport="rest",
)

SECRET_NAME = "projects/demo-project/secrets/app-db-credentials"
payload = {"username": "appuser", "password": "S3cureP@ss!", "host": "db.example.com"}

def get_secret(name):
    # TODO: Add try-except block to handle NotFound and return empty dict when secret not found
    response = client.access_secret_version(name=name + "/versions/latest")
    data = response.payload.data.decode("UTF-8")
    return json.loads(data)

if __name__ == "__main__":
    # Create or update the secret (idempotent)
    try:
        client.create_secret(
            parent="projects/demo-project",
            secret_id="app-db-credentials",
            secret={"replication": {"automatic": {}}}
        )
        client.add_secret_version(
            parent=SECRET_NAME,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Created:", SECRET_NAME)
    except Exception:
        # Assume secret exists, so add a new version
        client.add_secret_version(
            parent=SECRET_NAME,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Updated:", SECRET_NAME)

    # Test retrieving an existing secret (should work)
    secret = get_secret(SECRET_NAME)
    print("Secret host:", secret.get("host", "N/A"))

    # Test retrieving a non-existent secret (should be handled gracefully)
    missing_secret = get_secret("projects/demo-project/secrets/nonexistent-secret")
    print("Missing secret result:", missing_secret)

```

In a production environment, you cannot assume that external resources will always be available. Network issues, accidental deletions, or configuration errors can cause your application to fail. By catching specific exceptions like `NotFound`, you prevent a single missing secret from crashing your entire service.

Here is the updated `main.py` with the requested error handling.

---

### Updated `main.py`

```python
import json
from google.auth.credentials import AnonymousCredentials
from google.api_core.client_options import ClientOptions
from google.cloud import secretmanager
# FIX: Imported NotFound from google.api_core.exceptions
from google.api_core.exceptions import NotFound

client = secretmanager.SecretManagerServiceClient(
    credentials=AnonymousCredentials(),
    client_options=ClientOptions(api_endpoint="http://localhost:8080"),
    transport="rest",
)

SECRET_NAME = "projects/demo-project/secrets/app-db-credentials"
payload = {"username": "appuser", "password": "S3cureP@ss!", "host": "db.example.com"}

def get_secret(name):
    # FIX: Added try-except block to handle missing secrets
    try:
        response = client.access_secret_version(name=name + "/versions/latest")
        data = response.payload.data.decode("UTF-8")
        return json.loads(data)
    except NotFound:
        print("Secret not found")
        return {}

if __name__ == "__main__":
    # Create or update the secret (idempotent)
    try:
        client.create_secret(
            parent="projects/demo-project",
            secret_id="app-db-credentials",
            secret={"replication": {"automatic": {}}}
        )
        client.add_secret_version(
            parent=SECRET_NAME,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Created:", SECRET_NAME)
    except Exception:
        # Assume secret exists, so add a new version
        client.add_secret_version(
            parent=SECRET_NAME,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Updated:", SECRET_NAME)

    # Test retrieving an existing secret (should work)
    secret = get_secret(SECRET_NAME)
    print("Secret host:", secret.get("host", "N/A"))

    # Test retrieving a non-existent secret (should be handled gracefully)
    missing_secret = get_secret("projects/demo-project/secrets/nonexistent-secret")
    print("Missing secret result:", missing_secret)
```

---

### Understanding the Flow of Error Handling



When dealing with cloud APIs, it is important to distinguish between different types of failures. Here is why we use `NotFound`:

1.  **Specificity**: By catching `NotFound`, we only handle the case where the resource is missing. If there is a different error (like a `PermissionDenied` error), the program will still raise an exception, alerting you that your service account doesn't have the right access.
2.  **Graceful Degradation**: By returning an empty dictionary `{}`, your application can continue to run. Using `.get("host", "N/A")` on that dictionary ensures your code doesn't throw a `KeyError` later on.
3.  **The "Late Bind" Pattern**: In many microservice architectures, secrets might be created by a different process. Graceful handling allows your service to start up and wait for the secret to become available rather than entering a "crash loop."

### Why this is essential
Without this block, calling `get_secret` on a missing resource would result in a `google.api_core.exceptions.NotFound: 404` error, stopping your code execution immediately. This is particularly dangerous in background tasks or long-running web servers.

## Validating Retrieved Secrets for Required Fields and Data Integrity

Now that you've learned how to securely create and retrieve secrets using Google Cloud Secret Manager, it's time to build a robust workflow that ensures your application's data integrity.

Your task is to add validation logic after retrieving the secret in main.py:

    Check that all three required fields are present: "username", "password", and "host"
    Ensure each field contains actual data (not empty strings)
    Print "Secret validation successful - all fields present" if all checks pass
    Print "Validation failed: [fieldname] is missing or empty" for any missing or empty field (replace [fieldname] with the actual field name)

This exercise will help you build reliable applications that verify their data at every step, which is essential for production systems.

```python
import json
from google.auth.credentials import AnonymousCredentials
from google.api_core.client_options import ClientOptions
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient(
    credentials=AnonymousCredentials(),
    client_options=ClientOptions(api_endpoint="http://localhost:8080"),
    transport="rest",
)

SECRET_NAME = "projects/demo-project/secrets/app-db-credentials"
payload = {"username": "appuser", "password": "S3cureP@ss!", "host": "db.example.com"}

def create_or_update_secret():
    parent = SECRET_NAME
    try:
        client.create_secret(
            parent="/".join(parent.split("/")[:-2]),
            secret_id=parent.split("/")[-1],
            secret={"replication": {"automatic": {}}}
        )
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Created:", parent)
    except Exception:
        response = client.add_secret_version(
            parent=parent,
            payload={"data": json.dumps(payload).encode("UTF-8")}
        )
        print("Updated:", parent)

def get_secret(secret_name):
    response = client.access_secret_version(name=secret_name + "/versions/latest")
    return json.loads(response.payload.data.decode("UTF-8"))

if __name__ == "__main__":
    create_or_update_secret()
    
    # Retrieve the secret
    secret = get_secret(SECRET_NAME)
    
    # TODO: Validate that all required fields ("username", "password", "host") are present and non-empty
    # TODO: Print "Secret validation successful - all fields present" if all validations pass
    # TODO: Print "Validation failed: [fieldname] is missing or empty" if any field is missing or empty

```

Menambahkan logika validasi data sangatlah krusial. Dalam sistem produksi, mengasumsikan bahwa data dari *Secret Manager* selalu lengkap adalah cara cepat untuk mendapatkan `KeyError` di tengah malam.

Berikut adalah pembaruan pada blok `if __name__ == "__main__":` untuk mengimplementasikan validasi yang Anda minta:

```python
if __name__ == "__main__":
    create_or_update_secret()
    
    # Retrieve the secret
    secret = get_secret(SECRET_NAME)
    
    # List fields yang diwajibkan
    required_fields = ["username", "password", "host"]
    all_valid = True

    for field in required_fields:
        # Cek apakah key ada di dictionary dan nilainya tidak kosong/hanya whitespace
        value = secret.get(field)
        if not value or (isinstance(value, str) and not value.strip()):
            print(f"Validation failed: {field} is missing or empty")
            all_valid = False
            # Opsi: gunakan 'break' jika ingin berhenti di kesalahan pertama
    
    if all_valid:
        print("Secret validation successful - all fields present")
```

### Mengapa Pendekatan Ini Efektif?

1.  **Metode `.get()`**: Kita menggunakan `secret.get(field)` daripada `secret[field]`. Ini mencegah program *crash* dengan `KeyError` jika *key* tersebut memang tidak ada.
2.  **Falsy Checking**: Di Python, `if not value` akan bernilai `True` jika nilainya adalah `None` atau string kosong `""`.
3.  **`.strip()` (Opsional tapi Disarankan)**: Penggunaan `.strip()` memastikan bahwa string yang hanya berisi spasi (misal: `" "`) juga dianggap tidak valid, karena spasi tersebut biasanya bukan *host* atau *password* yang sah.
4.  **Skalabilitas**: Jika di masa depan Anda perlu menambahkan field baru (misal: `"port"`), Anda cukup menambahkannya ke dalam list `required_fields`.



### Tips Pro untuk Produksi
Dalam skenario nyata, jika validasi gagal, biasanya Anda ingin menghentikan eksekusi program sepenuhnya menggunakan `raise Exception` atau `sys.exit()`. Menjalankan aplikasi dengan kredensial database yang tidak lengkap hanya akan menyebabkan error yang lebih sulit dilacak di lapisan koneksi database nantinya.

## Validating Retrieved Secrets for Required Fields and Data Integrity